# 02 - Transform

**Stage:** Transform (the *T* in ETL).

Responsibilities:
- Load raw data from `data/raw/`.
- Reshape the table into the **right width and height**:
  - *Width*: select/engineer the correct set of columns (features + target).
  - *Height*: drop duplicates / invalid rows so each row is one observation.
- Clean types, handle missing values, engineer features.
- Persist a tidy, analysis-ready table to `data/interim/`.

In [ ]:
# Make the project `src` package importable from the notebooks/ dir.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

config.ensure_dirs()

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(config.RAW_FILE)
print(f"Loaded raw shape: {df.shape}")

## 1. Fix the height — one clean row per observation

In [ ]:
# Remove exact duplicate rows.
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows; new height = {len(df)}")

## 2. Fix the width — select & engineer columns

In [ ]:
# Standardise column names: lower_snake_case.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^0-9a-z]+", "_", regex=True)
    .str.strip("_")
)
df.columns.tolist()

In [ ]:
# Example feature engineering. Adjust to your real schema.
# If raw distance-to-beach exists, derive the target label.
if "distance_to_beach_m" in df.columns and config.TARGET_COLUMN not in df.columns:
    df[config.TARGET_COLUMN] = (df["distance_to_beach_m"] <= 1000).astype(int)

df.head()

## 3. Type coercion & missing values

In [ ]:
# Coerce obvious numeric columns; report any conversion failures.
numeric_candidates = df.select_dtypes(include="object").columns
for col in numeric_candidates:
    coerced = pd.to_numeric(df[col], errors="coerce")
    # Only adopt the coercion if it doesn't destroy most of the data.
    if coerced.notna().mean() > 0.9:
        df[col] = coerced

df.dtypes

In [ ]:
# Simple, transparent missing-value handling.
# Numeric -> median, categorical -> mode. Tune as appropriate.
for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode(dropna=True).iloc[0])

assert df.isna().sum().sum() == 0, "Unexpected remaining NaNs"

## 4. Validate final shape

In [ ]:
print(f"Transformed shape (height x width): {df.shape}")
df.describe(include="all").T

## 5. Persist transformed data

In [ ]:
out = config.TRANSFORMED_FILE
df.to_parquet(out, index=False)
print(f"Wrote transformed data -> {out}")